# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, en cumulant les résultats des 4 variantes.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` sauf en section ML · Python pur = affichage seulement

## 0 · Session Spark & imports

In [1]:
import sys
import os
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType, BinaryType
)

import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .master("local[2]")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/18 10:30:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.0


## 1 - Chemins & constantes

In [2]:
TRAIN_PATH   = "./data/Train/"
TEST_PATH    = "./data/Test/"
OUTPUT_PREDS = "./output/predictions/"
MODEL_PATH   = "./output/model/"
TARGET_SIZE  = (64, 64)

## 2 - Parsing

Ce bloc lit des images depuis un dossier et pour chaque image :
- Charge le contenu binaire du fichier via Spark
- Applique une fonction Python grâce à une UDF
    - ouvre l'image avec PIL
    - redimensionne en 64x64
    - transforme les pixels RGB en une liste de float (0-255)
- Extrait le nom du fichier et du dossier parent depuis le chemin du fichier
- Renvoie un DataFrame Spark avec 3 colonnes

In [3]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    # Fonction exécutée sur les workers
    # reçoit les octets bruts d'une image
    try:
        # import à l'intérieur de la fonction pour éviter les problèmes d'importation sur les workers
        from PIL import Image
        # décode les octets en images, force RGB (3 canaux)
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        # redimensionne en widthxheight (Lanczos = qualité de rééchantillonnage)
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        # récupère les octets bruts de l'image redimensionnée
        raw = img.tobytes()
        n = len(raw) # 64*64*3 = 12288 octets pour une image RGB de 64x64
        # convertit les octets en liste de pixels (entiers 0-255)
        pixels = list(struct.unpack(f"{n}B", raw))
        # renvoie un tuple (width, height, channels, pixels) pour Spark
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

# Schéma Spark décrivant le format du tuple renvoyé par decode_image_bytes
# Il faut dire explicitement à Spark le type de retour d'une fonction Python
_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

# Transforme la fonction Python en UDF Spark, avec le schéma de retour, utilisable dans une colonne Spark
decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        # lit le contenu des fichiers comme des octets bruts 
        spark.read.format("binaryFile")
        # va chercher les fichiers même dans les sous dossiers 
        .option("recursiveFileLookup", "true")
        # ne garde que les images (filtre par extension)
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
        .repartition(8)  # paralllélise le traitement sur plusieurs partitions
    )
    return (
        raw
        .select(
            # capture ce qui suit le dernier "/" --> nom du fichier
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            # capture le nom du dossier parent --> label 
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            # content est la colonne où binaryFile lit les octets bruts de l'image
            # nom donné par défaut par binaryFile, on la renomme en raw_bytes pour plus de clarté
            F.col("content").alias("raw_bytes"),
        )
        # applique l'UDF sur chaque ligne, résultat --> colonne "struct" avec width, height, channels, pixels
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        # enlève les lignes où l'image est illisible
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            # on ne garde que les pixels décodés
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show(5)

/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                

Images train : 624


/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                

Images test  : 408


+-----------+-------+--------------------+
|   image_id|  label|              pixels|
+-----------+-------+--------------------+
| 000077.png|tulipes|[252.0, 251.0, 25...|
| 000152.jpg|tulipes|[236.0, 116.0, 12...|
| 000063.jpg|tulipes|[178.0, 103.0, 94...|
|000049.jpeg|tulipes|[7.0, 7.0, 7.0, 9...|
| 000083.png|    lys|[139.0, 172.0, 11...|
+-----------+-------+--------------------+
only showing top 5 rows



## 2.2 - Parsing : normalisation couleur

In [4]:
def normalize_rgb(pixels):
    # Reçoit "pixels": liste de float représentant les valeurs RGB (0-255) 
    if pixels is None:
        # Sécurité : si l'image est illisible, on renvoie None
        return None
    # division de chaque pixel par 255.0 pour normaliser entre 0 et 1
    # appliqué élément par élément 
    return [p / 255.0 for p in pixels]

# convertit la fonction Python en UDF Spark, avec le type de retour ArrayType(FloatType())
normalize_rgb_udf = F.udf(normalize_rgb, ArrayType(FloatType()))

def preprocess_rgb(df):
    # ajoute une colonne "pixels_norm" avec les pixels normalisés entre 0 et 1
    return df.withColumn("pixels_norm", normalize_rgb_udf(F.col("pixels")))

# applique le preprocessing sur les DataFrames train et test
train_rgb_normalized_df = preprocess_rgb(train_parsed_df)
test_rgb_normalized_df  = preprocess_rgb(test_parsed_df)

print("Aperçu après normalisation RGB :")
train_rgb_normalized_df.select("image_id", "label", "pixels_norm").show(5, truncate=40)

# vérification rapide: taill eattendue = 64*54*3 = 12288 valeurs RGB normalisés 
expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_rgb_normalized_df
    # F.size() calcule la taille de la liste dans la colonne pixels_norm
    .select(F.size(F.col("pixels_norm")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_norm (attendu {expected_len}) : {check_len}")

Aperçu après normalisation RGB :


/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                

+----------+-------+----------------------------------------+
|  image_id|  label|                             pixels_norm|
+----------+-------+----------------------------------------+
|000162.jpg|tulipes|[0.72156864, 0.5882353, 0.69411767, 0...|
|000164.jpg|tulipes|[0.8784314, 0.49411765, 0.70980394, 0...|
|000128.jpg|    lys|[0.31764707, 0.76862746, 0.019607844,...|
|000123.png|    lys|[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1...|
|000038.png|tulipes|[0.827451, 0.5176471, 0.6901961, 0.69...|
+----------+-------+----------------------------------------+
only showing top 5 rows



/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[Stage 18:=============================================>          (18 + 2) / 22]

Taille pixels_norm (attendu 12288) : 12288


## 3 - Prétraitement niveaux de gris

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit.
Deux variantes de gris sont produites séparément :
- `pixels_grayscale` : niveaux de gris **non normalisés** (0-255), calculés à partir de `pixels` (RGB brut)
- `pixels_gray` : niveaux de gris **normalisés** ([0.0, 1.0]), calculés à partir de `pixels_norm` (RGB déjà normalisé)

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

In [5]:
def rgb_to_grayscale(pixels):
    # Reçoit pixels: liste de float représentant les valeurs RGB normalisées (0-1)
    if pixels is None:
        return None
    gray = []
    # on parcourt les pixels par groupes de 3 (R, G, B)
    for i in range(0, len(pixels), 3):
        # découpe le paquet en 3 variables nommées r, g, b 
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        # Combine les 3 canaux en une seule valeur de gris, pondérée selon la sensibilité
        # de l'œil humain (plus sensible au vert qu'au rouge, peu sensible au bleu)
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    # renvoie une liste 3x plus courte que l'entrée: un seul niveau de gris par pixel
    return gray

gray_udf = F.udf(rgb_to_grayscale, ArrayType(FloatType()))

def preprocess_grayscale(df):
    # à partir de "pixels" (RGB brut 0-255) -> gris non normalisé (0-255)
    return df.withColumn("pixels_grayscale", gray_udf(F.col("pixels")))

train_preprocessed_gray_df = preprocess_grayscale(train_parsed_df)
test_preprocessed_gray_df  = preprocess_grayscale(test_parsed_df)

print("Aperçu après prétraitement (gris non normalisé) :")
train_preprocessed_gray_df.select("image_id", "label", "pixels_grayscale").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_gray_df
    .select(F.size(F.col("pixels_grayscale")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_grayscale (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement (gris non normalisé) :


/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                

+----------+-------+----------------------------------------+
|  image_id|  label|                        pixels_grayscale|
+----------+-------+----------------------------------------+
|000162.jpg|tulipes|[163.244, 149.891, 130.483, 136.26, 1...|
|000164.jpg|tulipes|[161.572, 143.404, 131.025, 166.159, ...|
|000128.jpg|    lys|[139.841, 142.004, 125.616, 129.611, ...|
|000123.png|    lys|[255.0, 255.0, 255.0, 255.0, 255.0, 2...|
|000038.png|tulipes|[160.637, 117.111, 134.785, 150.524, ...|
+----------+-------+----------------------------------------+
only showing top 5 rows



/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[Stage 24:================================================>       (19 + 2) / 22]

Taille pixels_grayscale (attendu 4096) : 4096


In [6]:
def rgb_to_normalized_gray(pixels_norm):
    # Reçoit pixels: liste de float représentant les valeurs RGB normalisées (0-1)
    if pixels_norm is None:
        return None
    gray = []
    # on parcourt les pixels par groupes de 3 (R, G, B)
    for i in range(0, len(pixels_norm), 3):
        # découpe le paquet en 3 variables nommées r, g, b
        r, g, b = pixels_norm[i], pixels_norm[i + 1], pixels_norm[i + 2]
        # formule standard de luminance perceptuelle
        # Combine les 3 canaux en une seule valeur de gris, pondérée selon la sensibilité
        # de l'œil humain (plus sensible au vert qu'au rouge, peu sensible au bleu)
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    # Renvoie une liste 3x plus courte que l'entrée : un seul niveau de gris par pixel 
    return gray

norm_gray_udf = F.udf(rgb_to_normalized_gray, ArrayType(FloatType()))

def preprocess_normalized_gray(df):
    # à partir de "pixels_norm" (RGB déjà normalisé 0-1) -> gris normalisé (0-1)
    return df.withColumn("pixels_gray", norm_gray_udf(F.col("pixels_norm")))

train_preprocessed_norm_gray_df = preprocess_normalized_gray(train_rgb_normalized_df)
test_preprocessed_norm_gray_df  = preprocess_normalized_gray(test_rgb_normalized_df)

print("Aperçu après prétraitement (gris normalisé) :")
train_preprocessed_norm_gray_df.select("image_id", "label", "pixels_gray").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_norm_gray_df
    .select(F.size(F.col("pixels_gray")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_gray (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement (gris normalisé) :


/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                

+----------+-------+----------------------------------------+
|  image_id|  label|                             pixels_gray|
+----------+-------+----------------------------------------+
|000162.jpg|tulipes|[0.64017254, 0.58780783, 0.51169807, ...|
|000164.jpg|tulipes|[0.6336157, 0.56236863, 0.5138235, 0....|
|000128.jpg|    lys|[0.54839605, 0.55687845, 0.49261177, ...|
|000123.png|    lys|[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1...|
|000038.png|tulipes|[0.62994903, 0.45925882, 0.5285686, 0...|
+----------+-------+----------------------------------------+
only showing top 5 rows



/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[Stage 30:=============================================>          (18 + 2) / 22]

Taille pixels_gray (attendu 4096) : 4096


## Schéma commun des résultats (4 variantes)

In [7]:
results_schema = StructType([
    StructField("image_id", StringType(), False),
    StructField("true_label", StringType(), False),
    StructField("variant", StringType(), False),
    StructField("predicted_label", StringType(), False),
    StructField("confidence", FloatType(), False),
    StructField("prob_lys", FloatType(), False),
    StructField("prob_tulipes", FloatType(), False),
    StructField("pixels", BinaryType(), False),
])

os.makedirs(MODEL_PATH, exist_ok=True)

## ML couleurs - prétraitement en bytes

In [8]:
# Prétraitement couleur (bytes) : RGB brut 0-255, encodé en BinaryType
def pixels_to_bytes(pixels):
    if pixels is None:
        return None
    int_pixels = [int(p) for p in pixels]
    return struct.pack(f"{len(int_pixels)}B", *int_pixels)

bytes_udf = F.udf(pixels_to_bytes, BinaryType())

def preprocess_color_bytes(df):
    return df.withColumn("pixels_color_bytes", bytes_udf(F.col("pixels")))

train_color_bytes_df = preprocess_color_bytes(train_parsed_df)
test_color_bytes_df  = preprocess_color_bytes(test_parsed_df)

print("Aperçu après prétraitement :")
train_color_bytes_df.select("image_id", "label", "pixels_color_bytes").show(5, truncate=40)

expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_color_bytes_df
    .select(F.length(F.col("pixels_color_bytes")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_color_bytes en octets (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :


/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                

+----------+-------+----------------------------------------+
|  image_id|  label|                      pixels_color_bytes|
+----------+-------+----------------------------------------+
|000162.jpg|tulipes|[B8 96 B1 A9 8A A1 99 74 92 A3 77 9B ...|
|000164.jpg|tulipes|[E0 7E B5 CD 6D 9F C0 63 88 D1 96 89 ...|
|000128.jpg|    lys|[51 C4 05 50 C8 06 44 B2 07 44 B9 06 ...|
|000123.png|    lys|[FF FF FF FF FF FF FF FF FF FF FF FF ...|
|000038.png|tulipes|[D3 84 B0 B0 55 80 C8 63 94 D2 76 A2 ...|
+----------+-------+----------------------------------------+
only showing top 5 rows



/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[Stage 36:=============================================>          (18 + 2) / 22]

Taille pixels_color_bytes en octets (attendu 12288) : 12288


In [9]:
# ML - Random Forest sur les features couleur (bytes)
# collect() autorisé uniquement en section ML, pour repasser en numpy/sklearn

def to_numpy_bytes(df, feature_col, label_col="label"):
    rows = df.select(feature_col, label_col).collect()
    X = np.array([
        
        list(struct.unpack(f"{len(r[feature_col])}B", r[feature_col]))
        for r in rows
    ], dtype=np.float32)
    y = np.array([r[label_col] for r in rows])
    return X, y

X_train, y_train = to_numpy_bytes(train_color_bytes_df, "pixels_color_bytes")
X_test,  y_test  = to_numpy_bytes(test_color_bytes_df,  "pixels_color_bytes")

print(f"X_train : {X_train.shape}, X_test : {X_test.shape}")

param_dist = {
    "n_estimators": randint(50, 200),
    "max_depth": randint(5, 30),
    "min_samples_split": randint(2, 10),
}

rf = RandomForestClassifier(random_state=42)
search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=10, cv=3, random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)

best_rf = search.best_estimator_
print("Meilleurs hyperparamètres :", search.best_params_)

y_pred_color_bytes = best_rf.predict(X_test)

print(f"Accuracy  : {accuracy_score(y_test, y_pred_color_bytes):.3f}")
print(f"Precision : {precision_score(y_test, y_pred_color_bytes, average='weighted', zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_color_bytes, average='weighted', zero_division=0):.3f}")
print(f"F1        : {f1_score(y_test, y_pred_color_bytes, average='weighted', zero_division=0):.3f}")
print()
print(classification_report(y_test, y_pred_color_bytes, zero_division=0))

joblib.dump(best_rf, os.path.join(MODEL_PATH, "color_bytes.joblib"))

/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/Users/nina/big-data-spark/venv/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[Stage 44:==================================================>       (7 + 1) / 8]

X_train : (624, 12288), X_test : (408, 12288)


Meilleurs hyperparamètres : {'max_depth': 23, 'min_samples_split': 8, 'n_estimators': 124}
Accuracy  : 0.841
Precision : 0.842
Recall    : 0.841
F1        : 0.841

              precision    recall  f1-score   support

         lys       0.86      0.81      0.84       205
     tulipes       0.82      0.87      0.84       203

    accuracy                           0.84       408
   macro avg       0.84      0.84      0.84       408
weighted avg       0.84      0.84      0.84       408



['./output/model/color_bytes.joblib']

In [10]:
# Reconversion des résultats en DataFrame Spark (schéma commun aux 4 variantes)
y_proba_cb = best_rf.predict_proba(X_test)
classes_cb = [str(c) for c in best_rf.classes_]

results_rows_cb = []
for i, img_row in enumerate(test_color_bytes_df.select("image_id", "label", "pixels_color_bytes").collect()):
    proba = dict(zip(classes_cb, [float(p) for p in y_proba_cb[i]]))
    results_rows_cb.append((
        str(img_row["image_id"]),
        str(img_row["label"]),
        "color_bytes",
        str(y_pred_color_bytes[i]),
        float(max(proba.values())),
        float(proba.get("lys", 0.0)),
        float(proba.get("tulipes", 0.0)),
        bytes(img_row["pixels_color_bytes"]),
    ))

results_color_bytes_df = spark.createDataFrame(results_rows_cb, schema=results_schema)
results_color_bytes_df.show(5, truncate=40)

# Première écriture : initialise le fichier (écrase tout contenu précédent)
results_color_bytes_df.write.mode("overwrite").parquet(OUTPUT_PREDS)
print("Résultats color_bytes écrits dans", OUTPUT_PREDS)

26/07/18 10:32:11 WARN TaskSetManager: Stage 48 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.


+-----------+----------+-----------+---------------+----------+-----------+------------+----------------------------------------+
|   image_id|true_label|    variant|predicted_label|confidence|   prob_lys|prob_tulipes|                                  pixels|
+-----------+----------+-----------+---------------+----------+-----------+------------+----------------------------------------+
| 000077.png|   tulipes|color_bytes|        tulipes|0.75800693| 0.24199308|  0.75800693|[FC FB FA EB E7 E2 EC E7 E3 ED E8 E3 ...|
| 000152.jpg|   tulipes|color_bytes|        tulipes|0.94272274|0.057277266|  0.94272274|[EC 74 7C E1 A3 93 D9 E6 64 E0 DE 7F ...|
| 000063.jpg|   tulipes|color_bytes|        tulipes|0.78467745| 0.21532258|  0.78467745|[B2 67 5E E7 70 7B D4 51 55 DE 71 7E ...|
|000049.jpeg|   tulipes|color_bytes|        tulipes|0.57857686| 0.42142317|  0.57857686|[07 07 07 09 0B 05 09 0A 04 0A 0B 05 ...|
| 000083.png|       lys|color_bytes|            lys| 0.5380824|  0.5380824|  0.46191755|[8

26/07/18 10:32:12 WARN TaskSetManager: Stage 49 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.


Résultats color_bytes écrits dans ./output/predictions/


## ML couleurs normalisées

In [11]:
FEATURE_COL = "pixels_norm"
LABEL_COL   = "label"

def to_numpy(df, feature_col, label_col=LABEL_COL):
    # feature_col n'a pas de valeur par défaut car il peut varier selon le type de features (RGB normalisé, gris normalisé, etc.)
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

# suffixe "_cn" = "color normalized" pour distinguer l'expérience
train_ids_cn, X_train_cn, y_train_raw_cn = to_numpy(train_rgb_normalized_df, FEATURE_COL)
test_ids_cn,  X_test_cn,  y_test_raw_cn  = to_numpy(test_rgb_normalized_df, FEATURE_COL)

print(f"X_train shape : {X_train_cn.shape}")
print(f"X_test  shape : {X_test_cn.shape}")

label_encoder_cn = LabelEncoder()
# encodeur propre à cette expérience 
y_train_cn = label_encoder_cn.fit_transform(y_train_raw_cn)
y_test_cn  = label_encoder_cn.transform(y_test_raw_cn)
print(f"Classes : {dict(enumerate(label_encoder_cn.classes_))}")

rf_rgb_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_rgb_model.fit(X_train_cn, y_train_cn)

y_pred_cn       = rf_rgb_model.predict(X_test_cn)
y_pred_proba_cn = rf_rgb_model.predict_proba(X_test_cn)

acc_cn = accuracy_score(y_test_cn, y_pred_cn)
print(f"\nAccuracy (couleurs normalisées) : {acc_cn:.4f}")
print(classification_report(y_test_cn, y_pred_cn, target_names=label_encoder_cn.classes_))

# sauvegarde du modèle et de l'encodeur pour réutilisation l'étape d'inférence
joblib.dump(rf_rgb_model, os.path.join(MODEL_PATH, "color_normalized.joblib"))
# Sérialise l'objet Python rf_rgb_model (avec tous ses arbres entraînés) dans un fichier .joblib,
# pour pouvoir le recharger plus tard sans avoir à ré-entraîner (ex: dans Streamlit)

joblib.dump(label_encoder_cn, os.path.join(MODEL_PATH, "color_normalized_encoder.joblib"))
# Sauvegarde AUSSI l'encodeur de labels — indispensable pour retraduire les prédictions
# numériques (0, 1) en noms de classes lisibles ("lys", "tulipes") au moment de l'inférence,
# sans ça on ne saurait plus quel entier correspond à quelle fleur

X_train shape : (624, 12288)
X_test  shape : (408, 12288)
Classes : {0: np.str_('lys'), 1: np.str_('tulipes')}

Accuracy (couleurs normalisées) : 0.8505
              precision    recall  f1-score   support

         lys       0.88      0.81      0.85       205
     tulipes       0.83      0.89      0.86       203

    accuracy                           0.85       408
   macro avg       0.85      0.85      0.85       408
weighted avg       0.85      0.85      0.85       408



['./output/model/color_normalized_encoder.joblib']

In [12]:
# Reconversion des résultats en DataFrame Spark (schéma commun)
predicted_labels_cn = label_encoder_cn.inverse_transform(y_pred_cn)
classes_cn = [str(c) for c in label_encoder_cn.classes_]

pixel_rows_cn = {
    r["image_id"]: r["pixels"]
    for r in test_rgb_normalized_df.select("image_id", "pixels").collect()
}

results_rows_cn = []
for i, img_id in enumerate(test_ids_cn):
    proba = dict(zip(classes_cn, [float(p) for p in y_pred_proba_cn[i]]))
    raw_pixels = pixel_rows_cn[img_id]
    pixel_bytes = struct.pack(f"{len(raw_pixels)}B", *[int(p) for p in raw_pixels])
    results_rows_cn.append((
        str(img_id), str(y_test_raw_cn[i]), "color_normalized",
        str(predicted_labels_cn[i]), float(max(proba.values())),
        float(proba.get("lys", 0.0)), float(proba.get("tulipes", 0.0)),
        pixel_bytes,
    ))

results_color_norm_df = spark.createDataFrame(results_rows_cn, schema=results_schema)
results_color_norm_df.show(5, truncate=40)

results_color_norm_df.write.mode("append").parquet(OUTPUT_PREDS)
print("Résultats color_normalized ajoutés à", OUTPUT_PREDS)

26/07/18 10:32:46 WARN TaskSetManager: Stage 59 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.
26/07/18 10:32:47 WARN TaskSetManager: Stage 60 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.


+-----------+----------+----------------+---------------+----------+--------+------------+----------------------------------------+
|   image_id|true_label|         variant|predicted_label|confidence|prob_lys|prob_tulipes|                                  pixels|
+-----------+----------+----------------+---------------+----------+--------+------------+----------------------------------------+
| 000077.png|   tulipes|color_normalized|        tulipes|     0.655|   0.345|       0.655|[FC FB FA EB E7 E2 EC E7 E3 ED E8 E3 ...|
| 000152.jpg|   tulipes|color_normalized|        tulipes|     0.915|   0.085|       0.915|[6E 66 38 61 58 30 43 57 1B 45 65 1E ...|
| 000063.jpg|   tulipes|color_normalized|        tulipes|    0.7625|  0.2375|      0.7625|[9A 02 01 8C 02 03 78 00 02 7F 02 03 ...|
|000049.jpeg|   tulipes|color_normalized|        tulipes|    0.5425|  0.4575|      0.5425|[07 07 07 09 0B 05 09 0A 04 0A 0B 05 ...|
| 000083.png|       lys|color_normalized|            lys|      0.56|    0.56

## ML grayscale

In [13]:
FEATURE_COL = "pixels_grayscale"
LABEL_COL   = "label"

def to_numpy_gray(df, feature_col, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

train_ids_g, X_train_g, y_train_raw_g = to_numpy_gray(train_preprocessed_gray_df, FEATURE_COL)
test_ids_g,  X_test_g,  y_test_raw_g  = to_numpy_gray(test_preprocessed_gray_df, FEATURE_COL)

print(f"X_train shape : {X_train_g.shape}")
print(f"X_test  shape : {X_test_g.shape}")

label_encoder_g = LabelEncoder()
y_train_g = label_encoder_g.fit_transform(y_train_raw_g)
y_test_g  = label_encoder_g.transform(y_test_raw_g)
print(f"Classes : {dict(enumerate(label_encoder_g.classes_))}")

rf_gray_raw = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_gray_raw.fit(X_train_g, y_train_g)

y_pred_g       = rf_gray_raw.predict(X_test_g)
y_pred_proba_g = rf_gray_raw.predict_proba(X_test_g)

print(f"\nAccuracy (grayscale non normalisé) : {accuracy_score(y_test_g, y_pred_g):.4f}")
print(classification_report(y_test_g, y_pred_g, target_names=label_encoder_g.classes_))

joblib.dump(rf_gray_raw, os.path.join(MODEL_PATH, "grayscale.joblib"))
joblib.dump(label_encoder_g, os.path.join(MODEL_PATH, "grayscale_encoder.joblib"))

X_train shape : (624, 4096)
X_test  shape : (408, 4096)
Classes : {0: np.str_('lys'), 1: np.str_('tulipes')}

Accuracy (grayscale non normalisé) : 0.8529
              precision    recall  f1-score   support

         lys       0.90      0.80      0.85       205
     tulipes       0.82      0.91      0.86       203

    accuracy                           0.85       408
   macro avg       0.86      0.85      0.85       408
weighted avg       0.86      0.85      0.85       408



['./output/model/grayscale_encoder.joblib']

In [14]:
predicted_labels_g = label_encoder_g.inverse_transform(y_pred_g)
classes_g = [str(c) for c in label_encoder_g.classes_]

pixel_rows_g = {
    r["image_id"]: r["pixels"]
    for r in test_preprocessed_gray_df.select("image_id", "pixels").collect()
}

results_rows_g = []
for i, img_id in enumerate(test_ids_g):
    proba = dict(zip(classes_g, [float(p) for p in y_pred_proba_g[i]]))
    raw_pixels = pixel_rows_g[img_id]
    pixel_bytes = struct.pack(f"{len(raw_pixels)}B", *[int(p) for p in raw_pixels])
    results_rows_g.append((
        str(img_id), str(y_test_raw_g[i]), "grayscale",
        str(predicted_labels_g[i]), float(max(proba.values())),
        float(proba.get("lys", 0.0)), float(proba.get("tulipes", 0.0)),
        pixel_bytes,
    ))

results_gray_df = spark.createDataFrame(results_rows_g, schema=results_schema)
results_gray_df.show(5, truncate=40)

results_gray_df.write.mode("append").parquet(OUTPUT_PREDS)
print("Résultats grayscale ajoutés à", OUTPUT_PREDS)

+-----------+----------+---------+---------------+----------+--------+------------+----------------------------------------+
|   image_id|true_label|  variant|predicted_label|confidence|prob_lys|prob_tulipes|                                  pixels|
+-----------+----------+---------+---------------+----------+--------+------------+----------------------------------------+
| 000077.png|   tulipes|grayscale|        tulipes|      0.75|    0.25|        0.75|[FC FB FA EB E7 E2 EC E7 E3 ED E8 E3 ...|
| 000152.jpg|   tulipes|grayscale|        tulipes|     0.935|   0.065|       0.935|[6E 66 38 61 58 30 43 57 1B 45 65 1E ...|
| 000063.jpg|   tulipes|grayscale|        tulipes|      0.79|    0.21|        0.79|[9A 02 01 8C 02 03 78 00 02 7F 02 03 ...|
|000049.jpeg|   tulipes|grayscale|        tulipes|      0.53|    0.47|        0.53|[07 07 07 09 0B 05 09 0A 04 0A 0B 05 ...|
| 000083.png|       lys|grayscale|            lys|     0.625|   0.625|       0.375|[8B AC 74 A2 BA 8F A2 BA 8D 94 B4 80 ...|


26/07/18 10:33:20 WARN TaskSetManager: Stage 70 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.
26/07/18 10:33:20 WARN TaskSetManager: Stage 71 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.


## ML grayscale normalisé

In [15]:
FEATURE_COL = "pixels_gray"
LABEL_COL   = "label"

def to_numpy_gray_norm(df, feature_col, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

train_ids_gn, X_train_gn, y_train_raw_gn = to_numpy_gray_norm(train_preprocessed_norm_gray_df, FEATURE_COL)
test_ids_gn,  X_test_gn,  y_test_raw_gn  = to_numpy_gray_norm(test_preprocessed_norm_gray_df, FEATURE_COL)

print(f"X_train_gray shape : {X_train_gn.shape}")
print(f"X_test_gray  shape : {X_test_gn.shape}")

label_encoder_gn = LabelEncoder()
label_encoder_gn.fit(y_train_raw_gn)
y_train_gn = label_encoder_gn.transform(y_train_raw_gn)
y_test_gn  = label_encoder_gn.transform(y_test_raw_gn)
print(f"Classes : {dict(enumerate(label_encoder_gn.classes_))}")

rf_gray_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_gray_model.fit(X_train_gn, y_train_gn)

y_pred_gn       = rf_gray_model.predict(X_test_gn)
y_pred_proba_gn = rf_gray_model.predict_proba(X_test_gn)

acc_gray_n = accuracy_score(y_test_gn, y_pred_gn)
print(f"\nAccuracy (grayscale normalisé) : {acc_gray_n:.4f}")
print(classification_report(y_test_gn, y_pred_gn, target_names=label_encoder_gn.classes_))

joblib.dump(rf_gray_model, os.path.join(MODEL_PATH, "grayscale_normalized.joblib"))
joblib.dump(label_encoder_gn, os.path.join(MODEL_PATH, "grayscale_normalized_encoder.joblib"))

X_train_gray shape : (624, 4096)
X_test_gray  shape : (408, 4096)
Classes : {0: np.str_('lys'), 1: np.str_('tulipes')}

Accuracy (grayscale normalisé) : 0.8529
              precision    recall  f1-score   support

         lys       0.90      0.80      0.85       205
     tulipes       0.82      0.91      0.86       203

    accuracy                           0.85       408
   macro avg       0.86      0.85      0.85       408
weighted avg       0.86      0.85      0.85       408



['./output/model/grayscale_normalized_encoder.joblib']

In [16]:
predicted_labels_gn = label_encoder_gn.inverse_transform(y_pred_gn)
classes_gn = [str(c) for c in label_encoder_gn.classes_]

pixel_rows_gn = {
    r["image_id"]: r["pixels"]
    for r in test_preprocessed_norm_gray_df.select("image_id", "pixels").collect()
}

results_rows_gn = []
for i, img_id in enumerate(test_ids_gn):
    proba = dict(zip(classes_gn, [float(p) for p in y_pred_proba_gn[i]]))
    raw_pixels = pixel_rows_gn[img_id]
    pixel_bytes = struct.pack(f"{len(raw_pixels)}B", *[int(p) for p in raw_pixels])
    results_rows_gn.append((
        str(img_id), str(y_test_raw_gn[i]), "grayscale_normalized",
        str(predicted_labels_gn[i]), float(max(proba.values())),
        float(proba.get("lys", 0.0)), float(proba.get("tulipes", 0.0)),
        pixel_bytes,
    ))

results_gray_norm_df = spark.createDataFrame(results_rows_gn, schema=results_schema)
results_gray_norm_df.show(5, truncate=40)

results_gray_norm_df.write.mode("append").parquet(OUTPUT_PREDS)
print("Résultats grayscale_normalized ajoutés à", OUTPUT_PREDS)

26/07/18 10:33:58 WARN TaskSetManager: Stage 81 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.
26/07/18 10:33:58 WARN TaskSetManager: Stage 82 contains a task of very large size (2469 KiB). The maximum recommended task size is 1000 KiB.


+-----------+----------+--------------------+---------------+----------+--------+------------+----------------------------------------+
|   image_id|true_label|             variant|predicted_label|confidence|prob_lys|prob_tulipes|                                  pixels|
+-----------+----------+--------------------+---------------+----------+--------+------------+----------------------------------------+
| 000077.png|   tulipes|grayscale_normalized|        tulipes|      0.75|    0.25|        0.75|[FC FB FA EB E7 E2 EC E7 E3 ED E8 E3 ...|
| 000152.jpg|   tulipes|grayscale_normalized|        tulipes|     0.935|   0.065|       0.935|[6E 66 38 61 58 30 43 57 1B 45 65 1E ...|
| 000063.jpg|   tulipes|grayscale_normalized|        tulipes|      0.79|    0.21|        0.79|[9A 02 01 8C 02 03 78 00 02 7F 02 03 ...|
|000049.jpeg|   tulipes|grayscale_normalized|        tulipes|      0.53|    0.47|        0.53|[07 07 07 09 0B 05 09 0A 04 0A 0B 05 ...|
| 000083.png|       lys|grayscale_normalized|   

## Résumé des 4 variantes

In [17]:
final_df = spark.read.parquet(OUTPUT_PREDS)

summary_df = (
    final_df
    .withColumn("correct", F.when(F.col("true_label") == F.col("predicted_label"), 1).otherwise(0))
    .groupBy("variant")
    .agg(
        F.count("*").alias("n_test"),
        F.avg("correct").alias("accuracy"),
        F.avg("confidence").alias("avg_confidence"),
    )
    .orderBy("variant")
)

summary_df.show(truncate=False)

+--------------------+------+------------------+------------------+
|variant             |n_test|accuracy          |avg_confidence    |
+--------------------+------+------------------+------------------+
|color_bytes         |408   |0.8406862745098039|0.7742671279930601|
|color_normalized    |408   |0.8504901960784313|0.786832925297466 |
|grayscale           |408   |0.8529411764705882|0.7888715286173072|
|grayscale_normalized|408   |0.8529411764705882|0.7888715286173072|
+--------------------+------+------------------+------------------+



In [18]:
summary_df = (
    final_df
    .withColumn("correct", F.when(F.col("true_label") == F.col("predicted_label"), 1).otherwise(0))
    .groupBy("variant")
    .agg(
        F.count("*").alias("n_test"),
        F.avg("correct").alias("accuracy"),
        F.avg("confidence").alias("avg_confidence"),
    )
)
summary_df.write.mode("overwrite").parquet("./output/summary/")